# 06_rnn_lstm_gru: Bidirectional Recurrent Sequence Classifiers
    
This notebook trains a recurrent classifier in PyTorch to classify sentence lengths (long vs. short) using vocabulary loaded from Gutenberg's *Alice in Wonderland*.


In [1]:
import re
import requests
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Load Alice in Wonderland from NLTK gutenberg corpus
import nltk
nltk.download('gutenberg', quiet=True)
from nltk.corpus import gutenberg
sentences_raw = gutenberg.sents('carroll-alice.txt')

cleaned_sentences = []
for s in sentences_raw:
    words = [w.lower() for w in s if re.match(r"^\w+$", w)]
    if 3 < len(words) < 25:
        cleaned_sentences.append(words)

# Build Vocabulary
vocab = {"<pad>": 0, "<unk>": 1}
for s in cleaned_sentences[:500]:
    for w in s:
        if w not in vocab:
            vocab[w] = len(vocab)
vocab_size = len(vocab)
print("Vocabulary Size:", vocab_size)

# Create Inputs (Pad sequences to length 20)
seq_len = 20
X_data = []
y_data = []

for s in cleaned_sentences[:300]:
    indices = [vocab.get(w, 1) for w in s]
    if len(indices) < seq_len:
        indices = indices + [0] * (seq_len - len(indices))
    else:
        indices = indices[:seq_len]
    X_data.append(indices)
    # Binary classification task: Sentence length > 12 tokens
    y_data.append(1 if len(s) > 12 else 0)

X = torch.tensor(X_data, dtype=torch.long)
y = torch.tensor(y_data, dtype=torch.long)

# 2. Define Model
embedding_dim = 16
hidden_dim = 24
num_classes = 2

class RecurrentClassifier(nn.Module):
    def __init__(self, cell_type="LSTM"):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        if cell_type == "RNN":
            self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        elif cell_type == "LSTM":
            self.rnn = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        elif cell_type == "GRU":
            self.rnn = nn.GRU(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
        
    def forward(self, x):
        embedded = self.embedding(x)
        out, _ = self.rnn(embedded)
        # Grab final step representation
        last_step = out[:, -1, :]
        return self.fc(last_step)

# 3. Train models
for cell_name in ["RNN", "LSTM", "GRU"]:
    model = RecurrentClassifier(cell_type=cell_name)
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()
    
    # Run 5 training epochs
    for epoch in range(5):
        logits = model(X)
        loss = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"{cell_name} Classifier trained successfully. Final Loss: {loss.item():.4f}")


Vocabulary Size: 1074


RNN Classifier trained successfully. Final Loss: 0.5859
LSTM Classifier trained successfully. Final Loss: 0.6165
GRU Classifier trained successfully. Final Loss: 0.6031


### Output Explanation
- The PyTorch script trains standard sequence modeling cells (RNN, LSTM, and GRU).
- Concatenating bidirectional sequence states provides contextual features from both directions to predict sentence properties.
